# 支持向量机（Support Vector Machine，SVM）

SVM 是在感知机基础上发展出来的强力分类算法：感知机只要找到**任意一个**可分超平面，而 SVM 要找**间隔最大**的那个超平面。

**三个层次：**
1. **硬间隔 SVM**：数据线性可分，找最大间隔超平面
2. **软间隔 SVM**：数据近似线性可分，允许少量误分类
3. **核 SVM**：数据线性不可分，用核函数映射到高维空间

## 1. 硬间隔 SVM：最大间隔超平面

### 1.1 几何间隔

样本点 $(x_i, y_i)$（$y_i \in \{+1, -1\}$）到超平面 $w \cdot x + b = 0$ 的**函数间隔**：

$$\hat{\gamma}_i = y_i(w \cdot x_i + b)$$

**几何间隔**（实际距离，除以 $\|w\|$ 归一化）：

$$\gamma_i = \frac{y_i(w \cdot x_i + b)}{\|w\|}$$

**间隔**（所有样本几何间隔的最小值）：$\gamma = \min_i \gamma_i$

### 1.2 优化目标

最大化间隔 → 等价于约束优化问题：

$$\min_{w,b} \frac{1}{2}\|w\|^2 \quad \text{s.t.} \quad y_i(w \cdot x_i + b) \geq 1, \; \forall i$$

- 约束：所有样本到超平面的函数间隔 $\geq 1$
- **支持向量**：恰好满足 $y_i(w \cdot x_i + b) = 1$ 的点，它们「顶住」了两侧边界
- 两侧支持向量之间的宽度 = $\frac{2}{\|w\|}$，最大化间隔等价于最小化 $\|w\|^2$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_blobs, make_circles, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 生成线性可分数据
X, y = make_blobs(n_samples=100, centers=2, cluster_std=0.8, random_state=42)
y = y * 2 - 1   # 转为 +1/-1

# 硬间隔 SVM（C 极大近似硬间隔）
svm_hard = SVC(kernel='linear', C=1e6)
svm_hard.fit(X, y)

def plot_svm_boundary(ax, clf, X, y, title):
    h = 0.02
    x_min, x_max = X[:,0].min()-1, X[:,0].max()+1
    y_min, y_max = X[:,1].min()-1, X[:,1].max()+1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, cmap='RdBu')

    ax.scatter(X[y==1,  0], X[y==1,  1], c='steelblue', edgecolors='k', s=40)
    ax.scatter(X[y==-1, 0], X[y==-1, 1], c='tomato',    edgecolors='k', s=40)

    # 决策边界与间隔线
    w   = clf.coef_[0]
    b   = clf.intercept_[0]
    x1  = np.linspace(x_min, x_max, 200)
    x2_mid = -(w[0]*x1 + b) / w[1]
    x2_pos = -(w[0]*x1 + b - 1) / w[1]
    x2_neg = -(w[0]*x1 + b + 1) / w[1]
    ax.plot(x1, x2_mid, 'k-',  linewidth=2, label='决策边界')
    ax.plot(x1, x2_pos, 'k--', linewidth=1, label='间隔边界')
    ax.plot(x1, x2_neg, 'k--', linewidth=1)

    # 支持向量
    sv = clf.support_vectors_
    ax.scatter(sv[:,0], sv[:,1], s=150, facecolors='none',
               edgecolors='black', linewidths=2, label=f'支持向量({len(sv)}个)')
    ax.set_title(title)
    ax.legend(fontsize=8)

fig, ax = plt.subplots(figsize=(7, 5))
plot_svm_boundary(ax, svm_hard, X, y, '硬间隔 SVM')
plt.tight_layout()
plt.show()
print(f"支持向量数量: {len(svm_hard.support_vectors_)}")
print(f"几何间隔宽度: {2 / np.linalg.norm(svm_hard.coef_):.4f}")

## 2. 软间隔 SVM：允许误分类

现实数据很少完全线性可分。软间隔引入**松弛变量** $\xi_i \geq 0$，允许样本违反间隔约束：

$$\min_{w,b,\xi} \frac{1}{2}\|w\|^2 + C \sum_{i=1}^{n} \xi_i$$

$$\text{s.t.} \quad y_i(w \cdot x_i + b) \geq 1 - \xi_i, \quad \xi_i \geq 0, \; \forall i$$

**参数 C（惩罚系数）**的意义：
- **C 大**：严格惩罚误分类，间隔变小，容易过拟合
- **C 小**：宽松对待误分类，间隔变大，更正则化，容易欠拟合

等价的 **Hinge Loss**（合页损失）：

$$L_{hinge}(y, f(x)) = \max(0, 1 - y \cdot f(x))$$

In [ ]:
# 含噪声的数据，演示不同 C 的效果
np.random.seed(0)
X_noisy, y_noisy = make_blobs(n_samples=120, centers=2, cluster_std=1.5, random_state=0)
y_noisy = y_noisy * 2 - 1

C_values = [0.01, 1, 100]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, C in zip(axes, C_values):
    svm_c = SVC(kernel='linear', C=C)
    svm_c.fit(X_noisy, y_noisy)
    plot_svm_boundary(ax, svm_c, X_noisy, y_noisy,
                      f'C={C}  间隔宽={2/np.linalg.norm(svm_c.coef_):.2f}')

plt.suptitle('软间隔 SVM：C 越小间隔越宽（更正则化）', fontsize=13)
plt.tight_layout()
plt.show()

## 3. 核函数（Kernel Trick）

当数据**线性不可分**时，将数据映射到更高维空间，在高维空间中线性可分：

$$x \xrightarrow{\phi} \phi(x) \in \mathbb{R}^M \quad (M \gg n)$$

**核技巧**：不需要显式计算 $\phi(x)$，只需计算核函数 $K(x_i, x_j) = \phi(x_i) \cdot \phi(x_j)$，避免高维计算开销。

常用核函数：

| 核函数 | 公式 | 参数 | 适用场景 |
|--------|------|------|----------|
| 线性核 | $K = x_i \cdot x_j$ | 无 | 线性可分、高维稀疏数据 |
| 多项式核 | $K = (\gamma x_i \cdot x_j + r)^d$ | $d, \gamma, r$ | 图像处理 |
| **RBF 高斯核** | $K = e^{-\gamma \|x_i - x_j\|^2}$ | $\gamma$ | 最常用，适合大多数场景 |
| Sigmoid 核 | $K = \tanh(\gamma x_i \cdot x_j + r)$ | $\gamma, r$ | 类似神经网络 |

In [ ]:
# 直观展示核函数将非线性数据变为可分
from mpl_toolkits.mplot3d import Axes3D

np.random.seed(42)
X_circ, y_circ = make_circles(n_samples=200, noise=0.1, factor=0.4, random_state=42)
y_circ = y_circ * 2 - 1

# RBF 核映射示意：手动添加一个新特征 r^2 = x1^2 + x2^2
r2 = X_circ[:,0]**2 + X_circ[:,1]**2

fig = plt.figure(figsize=(12, 4))

ax1 = fig.add_subplot(121)
ax1.scatter(X_circ[y_circ==1,  0], X_circ[y_circ==1,  1], c='steelblue', s=20, label='+1')
ax1.scatter(X_circ[y_circ==-1, 0], X_circ[y_circ==-1, 1], c='tomato',    s=20, label='-1')
ax1.set_title('原始空间（线性不可分）')
ax1.legend(fontsize=8)

ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(X_circ[y_circ==1,  0], X_circ[y_circ==1,  1], r2[y_circ==1],
            c='steelblue', s=20, label='+1')
ax2.scatter(X_circ[y_circ==-1, 0], X_circ[y_circ==-1, 1], r2[y_circ==-1],
            c='tomato',    s=20, label='-1')
ax2.set_xlabel('x1'); ax2.set_ylabel('x2'); ax2.set_zlabel('r²=x1²+x2²')
ax2.set_title('映射到 3D 后（线性可分）')
ax2.legend(fontsize=8)

plt.suptitle('核技巧：低维不可分 → 高维线性可分', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 四种核函数对同心圆数据的效果对比
kernels = [
    ('linear',  {}),
    ('poly',    {'degree': 3, 'gamma': 'auto'}),
    ('rbf',     {'gamma': 'scale'}),
    ('sigmoid', {'gamma': 'scale'}),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
h = 0.05
x_min, x_max = X_circ[:,0].min()-0.5, X_circ[:,0].max()+0.5
y_min, y_max = X_circ[:,1].min()-0.5, X_circ[:,1].max()+0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

for ax, (kernel, kw) in zip(axes, kernels):
    clf = SVC(kernel=kernel, C=1.0, **kw)
    clf.fit(X_circ, y_circ)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_circ[y_circ==1,  0], X_circ[y_circ==1,  1], c='steelblue', s=15)
    ax.scatter(X_circ[y_circ==-1, 0], X_circ[y_circ==-1, 1], c='tomato',    s=15)
    acc = accuracy_score(y_circ, clf.predict(X_circ))
    ax.set_title(f'kernel={kernel}\nacc={acc:.2%}')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('四种核函数对同心圆数据的效果', fontsize=13)
plt.tight_layout()
plt.show()

## 4. RBF 核的 gamma 参数

$$K(x_i, x_j) = e^{-\gamma \|x_i - x_j\|^2}$$

- **$\gamma$ 大**：核函数影响范围小，决策边界复杂，容易**过拟合**
- **$\gamma$ 小**：核函数影响范围大，决策边界平滑，容易**欠拟合**

C 和 gamma 共同控制模型复杂度，通常用**网格搜索**调参。

In [ ]:
gammas = [0.01, 0.1, 1, 10]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, gamma in zip(axes, gammas):
    clf = SVC(kernel='rbf', C=1.0, gamma=gamma)
    clf.fit(X_circ, y_circ)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_circ[y_circ==1,  0], X_circ[y_circ==1,  1], c='steelblue', s=15)
    ax.scatter(X_circ[y_circ==-1, 0], X_circ[y_circ==-1, 1], c='tomato',    s=15)
    ax.set_title(f'gamma={gamma}  sv={len(clf.support_vectors_)}')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('RBF 核：gamma 越大决策边界越弯曲（容易过拟合）', fontsize=13)
plt.tight_layout()
plt.show()

## 5. 对偶问题与 KKT 条件

原始问题通过拉格朗日乘子法转化为**对偶问题**：

$$\max_{\alpha} \sum_{i=1}^{n} \alpha_i - \frac{1}{2} \sum_{i,j} \alpha_i \alpha_j y_i y_j K(x_i, x_j)$$

$$\text{s.t.} \quad 0 \leq \alpha_i \leq C, \quad \sum_{i=1}^{n} \alpha_i y_i = 0$$

**KKT 条件（支持向量的判断依据）：**

| $\alpha_i$ | 样本位置 |
|-----------|----------|
| $\alpha_i = 0$ | 非支持向量，位于间隔之外，不参与决策 |
| $0 < \alpha_i < C$ | 支持向量，恰好在间隔边界上 |
| $\alpha_i = C$ | 软间隔中的误分类或间隔内样本 |

预测时只需支持向量参与计算：$f(x) = \text{sign}\left(\sum_{i \in SV} \alpha_i y_i K(x_i, x) + b\right)$

## 6. sklearn 实战：乳腺癌分类

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

data = load_breast_cancer()
X_bc, y_bc = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42)

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s  = sc.transform(X_te)

# 对比线性核与 RBF 核
for kernel in ['linear', 'rbf']:
    clf = SVC(kernel=kernel, C=1.0, random_state=42)
    clf.fit(X_tr_s, y_tr)
    y_pred = clf.predict(X_te_s)
    print(f"\n=== kernel='{kernel}' ===")
    print(f"支持向量数: {clf.n_support_}")
    print(classification_report(y_te, y_pred, target_names=data.target_names))

## 7. 网格搜索调参（C 和 gamma）

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C':     [0.01, 0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1, 'scale'],
}
grid = GridSearchCV(SVC(kernel='rbf', random_state=42),
                    param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_tr_s, y_tr)

print(f"最优参数: {grid.best_params_}")
print(f"交叉验证最优分数: {grid.best_score_:.4f}")
print(f"测试集准确率: {accuracy_score(y_te, grid.predict(X_te_s)):.4f}")

# 可视化 C vs gamma 的准确率热力图
scores = grid.cv_results_['mean_test_score']
scores = scores.reshape(len(param_grid['C']), len(param_grid['gamma']))

plt.figure(figsize=(8, 5))
sns.heatmap(scores, annot=True, fmt='.3f', cmap='YlGnBu',
            xticklabels=param_grid['gamma'],
            yticklabels=param_grid['C'])
plt.xlabel('gamma')
plt.ylabel('C')
plt.title('网格搜索：C × gamma 交叉验证准确率')
plt.tight_layout()
plt.show()

## 8. SVM 回归（SVR）

SVM 也可用于回归，称为 SVR（Support Vector Regression）。

引入 $\varepsilon$-不敏感损失：预测值在真实值 $\pm\varepsilon$ 范围内**不计损失**，超出才惩罚：

$$L_\varepsilon = \max(0, |y - f(x)| - \varepsilon)$$

- **$\varepsilon$**：允许的误差管道宽度，越大拟合越宽松
- **C**：超出管道的惩罚力度

In [ ]:
from sklearn.svm import SVR

np.random.seed(0)
X_reg = np.sort(np.random.rand(80) * 5).reshape(-1, 1)
y_reg = np.sin(X_reg).ravel() + np.random.randn(80) * 0.2
X_plot = np.linspace(0, 5, 200).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
configs = [
    ('rbf', 1.0,  0.2, 'RBF C=1  ε=0.2'),
    ('rbf', 10.0, 0.2, 'RBF C=10 ε=0.2'),
    ('rbf', 1.0,  0.5, 'RBF C=1  ε=0.5'),
]
for ax, (kernel, C, eps, label) in zip(axes, configs):
    svr = SVR(kernel=kernel, C=C, epsilon=eps)
    svr.fit(X_reg, y_reg)
    y_hat = svr.predict(X_plot)
    ax.scatter(X_reg, y_reg, color='gray', s=15, alpha=0.7)
    ax.plot(X_plot, y_hat, color='steelblue', linewidth=2, label=label)
    ax.fill_between(X_plot.ravel(), y_hat - eps, y_hat + eps,
                    alpha=0.2, color='steelblue', label=f'ε-管道(±{eps})')
    ax.set_title(label)
    ax.legend(fontsize=7)

plt.suptitle('SVR：ε-不敏感管道拟合', fontsize=13)
plt.tight_layout()
plt.show()

## 总结

| 知识点 | 核心内容 |
|--------|----------|
| 硬间隔 | $\min \frac{1}{2}\|w\|^2$，要求所有点在间隔外，仅适合完全可分 |
| 软间隔 | 引入松弛变量 $\xi_i$，C 控制容错程度 |
| 核函数 | 隐式映射到高维，$K(x_i,x_j)=\phi(x_i)\cdot\phi(x_j)$，RBF 最常用 |
| 支持向量 | 只有 $\alpha_i > 0$ 的点参与预测，模型稀疏 |
| 关键参数 | C（容错），gamma（RBF 带宽），$\varepsilon$（SVR 管道宽） |
| 调参 | GridSearchCV 对 C 和 gamma 网格搜索 |

**与感知机对比：**
| | 感知机 | SVM |
|--|--------|-----|
| 目标 | 找到任意可分超平面 | 找到最大间隔超平面 |
| 解唯一性 | 不唯一 | 唯一 |
| 线性不可分 | 不收敛 | 软间隔/核函数处理 |
| 泛化能力 | 弱 | 强（间隔越大泛化越好） |